# Visualizing the History of Nobel Prize Winners

The Nobel Prize has been among the most prestigious international awards since 1901. Each year, awards are bestowed in chemistry, literature, physics, physiology or medicine, economics, and peace. In addition to the honor, prestige, and substantial prize money, the recipient also gets a gold medal with an image of Alfred Nobel (1833 - 1896), who established the prize.

![](Nobel_Prize.png)

The Nobel Foundation has made a dataset available of all prize winners from the outset of the awards from 1901 to 2023. The dataset used in this project is from the Nobel Prize API and is available in the `nobel.csv` file in the `data` folder.

In this project, you'll get a chance to explore and answer several questions related to this prizewinning data. And we encourage you then to explore further questions that you're interested in!

Analyze Nobel Prize winner data and identify patterns by answering the following questions:

What is the most commonly awarded gender and birth country?

- Store your answers as string variables `top_gender` and `top_country`.

Which decade had the highest ratio of US-born Nobel Prize winners to total winners in all categories?

- Store this as an integer called `max_decade_usa`.

Which decade and Nobel Prize category combination had the highest proportion of female laureates?

- Store this as a dictionary called `max_female_dict` where the decade is the key and the category is the value. There should only be one key:value pair.

Who was the first woman to receive a Nobel Prize, and in what category?

- Save your string answers as `first_woman_name` and `first_woman_category`.

Which individuals or organizations have won more than one Nobel Prize throughout the years?

- Store the full names in a list named `repeat_list`.

In [11]:
# Loading in required libraries
import pandas as pd
import seaborn as sns
import numpy as np

# Start coding here!

### Loading the Nobel Prize Dataset

First, we'll load the `nobel.csv` file into a pandas DataFrame and inspect its first few rows and general information to understand the data structure and types.

In [12]:
nobel = pd.read_csv('data/nobel.csv')
display(nobel.head())
display(nobel.info())

,year,category,prize,motivation,prize_share,laureate_id,laureate_type,full_name,birth_date,birth_city,birth_country,sex,organization_name,organization_city,organization_country,death_date,death_city,death_country
0,1901,Chemistry,The Nobel Prize in Chemistry 1901,"""in recognition of the extraordinary services ...",1/1,160,Individual,Jacobus Henricus van 't Hoff,1852-08-30,Rotterdam,Netherlands,Male,Berlin University,Berlin,Germany,1911-03-01,Berlin,Germany
1,1901,Literature,The Nobel Prize in Literature 1901,"""in special recognition of his poetic composit...",1/1,569,Individual,Sully Prudhomme,1839-03-16,Paris,France,Male,NaN,NaN,NaN,1907-09-07,Châtenay,France
2,1901,Medicine,The Nobel Prize in Physiology or Medicine 1901,"""for his work on serum therapy, especially its...",1/1,293,Individual,Emil Adolf von Behring,1854-03-15,Hansdorf (Lawice),Prussia (Poland),Male,Marburg University,Marburg,Germany,1917-03-31,Marburg,Germany
3,1901,Peace,The Nobel Peace Prize 1901,NaN,1/2,462,Individual,Jean Henry Dunant,1828-05-08,Geneva,Switzerland,Male,NaN,NaN,NaN,1910-10-30,Heiden,Switzerland
4,1901,Peace,The Nobel Peace Prize 1901,NaN,1/2,463,Individual,Frédéric Passy,1822-05-20,Paris,France,Male,NaN,NaN,NaN,1912-06-12,Paris,France


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   year                  1000 non-null   int64 
 1   category              1000 non-null   object
 2   prize                 1000 non-null   object
 3   motivation            912 non-null    object
 4   prize_share           1000 non-null   object
 5   laureate_id           1000 non-null   int64 
 6   laureate_type         1000 non-null   object
 7   full_name             1000 non-null   object
 8   birth_date            968 non-null    object
 9   birth_city            964 non-null    object
 10  birth_country         969 non-null    object
 11  sex                   970 non-null    object
 12  organization_name     736 non-null    object
 13  organization_city     735 non-null    object
 14  organization_country  735 non-null    object
 15  death_date            596 non-null    o

None

### Question 1: Most Commonly Awarded Gender and Birth Country

To find the most commonly awarded gender and birth country, we'll count the occurrences of each unique value in the `gender` and `birth_country` columns, respectively, and then select the top one.

In [13]:
# Most commonly awarded gender
top_gender = nobel['sex'].value_counts().index[0]

# Most commonly awarded birth country
top_country = nobel['birth_country'].value_counts().index[0]

print(f"Most commonly awarded gender: {top_gender}")
print(f"Most commonly awarded birth country: {top_country}")

Most commonly awarded gender: Male
Most commonly awarded birth country: United States of America


### Question 2: Decade with Highest Ratio of US-born Nobel Prize Winners

To determine the decade with the highest ratio of US-born winners, we need to first extract the decade from the `year` column. Then, we'll calculate the number of US-born winners and total winners for each decade and compute their ratio.

In [14]:
# Create a 'decade' column
nobel['decade'] = (nobel['year'] // 10) * 10

# Filter for US-born winners
nobel_us_born = nobel[nobel['birth_country'] == 'United States of America']

# Count total winners per decade
total_winners_per_decade = nobel.groupby('decade').size()

# Count US-born winners per decade
us_born_winners_per_decade = nobel_us_born.groupby('decade').size()

# Calculate ratio of US-born to total winners per decade
ratio_us_to_total = us_born_winners_per_decade / total_winners_per_decade

# Find the decade with the highest ratio
max_decade_usa = ratio_us_to_total.idxmax()

print(f"Decade with highest ratio of US-born winners: {max_decade_usa}")

Decade with highest ratio of US-born winners: 2000


### Question 3: Decade and Category with Highest Proportion of Female Laureates

To find the decade and category with the highest proportion of female laureates, we will first filter for female laureates, then group by decade and category to count the number of female winners and total winners in each group, and finally calculate the proportion.

In [15]:
# Filter for female laureates
females = nobel[nobel['sex'] == 'Female']

# Group by decade and category, then count female winners
female_counts = females.groupby(['decade', 'category']).size().reset_index(name='female_winners')

# Group by decade and category, then count total winners
total_counts = nobel.groupby(['decade', 'category']).size().reset_index(name='total_winners')

# Merge the counts
merged_counts = pd.merge(female_counts, total_counts, on=['decade', 'category'], how='left')

# Calculate the proportion of female laureates
merged_counts['proportion_female'] = merged_counts['female_winners'] / merged_counts['total_winners']

# Find the row with the highest proportion
max_proportion_row = merged_counts.loc[merged_counts['proportion_female'].idxmax()]

# Store as a dictionary
max_female_dict = {int(max_proportion_row['decade']): max_proportion_row['category']}

print(f"Decade and category with highest proportion of female laureates: {max_female_dict}")

Decade and category with highest proportion of female laureates: {2020: 'Literature'}


### Question 4: First Woman to Receive a Nobel Prize and in What Category

To identify the first woman to receive a Nobel Prize and her category, we will filter the DataFrame for female laureates, sort them by `year` (ascending), and then extract the name and category of the first entry.

In [16]:
# Filter for female laureates and sort by year
first_woman_winner = nobel[nobel['sex'] == 'Female'].sort_values(by='year').iloc[0]

# Extract name and category
first_woman_name = first_woman_winner['full_name']
first_woman_category = first_woman_winner['category']

print(f"First woman to receive a Nobel Prize: {first_woman_name}")
print(f"Category: {first_woman_category}")

First woman to receive a Nobel Prize: Marie Curie, née Sklodowska
Category: Physics


### Question 5: Individuals or Organizations with Multiple Nobel Prizes

To find individuals or organizations who have won more than one Nobel Prize, we will group the data by `full_name` and count the number of awards for each. Then we will filter for those with more than one award.

In [17]:
# Count the number of awards per full_name
repeat_winners = nobel['full_name'].value_counts()

# Filter for those with more than one award
repeat_list = repeat_winners[repeat_winners > 1].index.tolist()

print(f"Individuals or organizations with multiple Nobel Prizes: {repeat_list}")

Individuals or organizations with multiple Nobel Prizes: ['Comité international de la Croix Rouge (International Committee of the Red Cross)', 'Linus Carl Pauling', 'John Bardeen', 'Frederick Sanger', 'Marie Curie, née Sklodowska', 'Office of the United Nations High Commissioner for Refugees (UNHCR)']
